In [554]:
import pandas as pd
import numpy as np
# import torch
from torch import tensor, from_numpy, nn, optim, float32, reshape
from torch.utils.data import TensorDataset, DataLoader
from torchvision import transforms
from sklearn.preprocessing import Normalizer, LabelBinarizer
from sklearn.model_selection import train_test_split

In [555]:
df = pd.read_csv("housing.csv")
df.head()

,longitude,latitude,housing_median_age,total_rooms,total_bedrooms,population,households,median_income,ocean_proximity,median_house_value
0,-117.61,34.13,21.0,8416.0,1386.0,4308.0,1341.0,4.4611,INLAND,164600.0
1,-117.37,33.98,52.0,201.0,44.0,130.0,24.0,2.0250,INLAND,125000.0
2,-118.34,33.89,36.0,2274.0,411.0,1232.0,423.0,5.3730,<1H OCEAN,244500.0
3,-118.92,35.13,29.0,1297.0,262.0,909.0,253.0,1.9236,INLAND,106300.0
4,-121.80,37.23,18.0,3179.0,526.0,1663.0,507.0,5.9225,<1H OCEAN,265800.0


In [556]:
df.isnull().any()

longitude             False
latitude              False
housing_median_age    False
total_rooms           False
total_bedrooms         True
population            False
households            False
median_income         False
ocean_proximity       False
median_house_value    False
dtype: bool

In [557]:
df_filled = df.fillna(method="bfill", axis=1)
df_filled

,longitude,latitude,housing_median_age,total_rooms,total_bedrooms,population,households,median_income,ocean_proximity,median_house_value
0,-117.61,34.13,21.0,8416.0,1386.0,4308.0,1341.0,4.4611,INLAND,164600.0
1,-117.37,33.98,52.0,201.0,44.0,130.0,24.0,2.025,INLAND,125000.0
2,-118.34,33.89,36.0,2274.0,411.0,1232.0,423.0,5.373,<1H OCEAN,244500.0
3,-118.92,35.13,29.0,1297.0,262.0,909.0,253.0,1.9236,INLAND,106300.0
4,-121.8,37.23,18.0,3179.0,526.0,1663.0,507.0,5.9225,<1H OCEAN,265800.0
...,...,...,...,...,...,...,...,...,...,...
16507,-119.53,36.55,34.0,2065.0,343.0,1041.0,313.0,3.2917,INLAND,111500.0
16508,-122.4,37.73,50.0,1947.0,411.0,1170.0,384.0,3.4769,NEAR BAY,238700.0
16509,-118.41,33.92,29.0,1436.0,401.0,674.0,343.0,3.6389,<1H OCEAN,275000.0
16510,-117.08,32.62,36.0,1674.0,309.0,818.0,307.0,3.4773,NEAR OCEAN,150400.0


In [558]:
binarizer = LabelBinarizer()
df_filled["ocean_proximity"] = binarizer.fit_transform(df_filled["ocean_proximity"])
df_filled

,longitude,latitude,housing_median_age,total_rooms,total_bedrooms,population,households,median_income,ocean_proximity,median_house_value
0,-117.61,34.13,21.0,8416.0,1386.0,4308.0,1341.0,4.4611,0,164600.0
1,-117.37,33.98,52.0,201.0,44.0,130.0,24.0,2.025,0,125000.0
2,-118.34,33.89,36.0,2274.0,411.0,1232.0,423.0,5.373,1,244500.0
3,-118.92,35.13,29.0,1297.0,262.0,909.0,253.0,1.9236,0,106300.0
4,-121.8,37.23,18.0,3179.0,526.0,1663.0,507.0,5.9225,1,265800.0
...,...,...,...,...,...,...,...,...,...,...
16507,-119.53,36.55,34.0,2065.0,343.0,1041.0,313.0,3.2917,0,111500.0
16508,-122.4,37.73,50.0,1947.0,411.0,1170.0,384.0,3.4769,0,238700.0
16509,-118.41,33.92,29.0,1436.0,401.0,674.0,343.0,3.6389,1,275000.0
16510,-117.08,32.62,36.0,1674.0,309.0,818.0,307.0,3.4773,0,150400.0


In [559]:
x = df_filled.drop("median_house_value", axis=1)
y = df_filled["median_house_value"]

x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.1)
x_train

,longitude,latitude,housing_median_age,total_rooms,total_bedrooms,population,households,median_income,ocean_proximity
15137,-121.91,37.31,16.0,2962.0,898.0,1555.0,795.0,2.5804,1
4793,-121.83,37.35,31.0,2914.0,715.0,3547.0,645.0,3.7143,1
4709,-124.16,40.8,52.0,2167.0,480.0,908.0,451.0,1.6111,0
7652,-121.8,39.75,28.0,2551.0,378.0,1011.0,374.0,4.3309,0
2500,-118.34,33.84,36.0,1407.0,231.0,676.0,231.0,5.269,1
...,...,...,...,...,...,...,...,...,...
698,-122.0,37.31,28.0,3811.0,585.0,1795.0,581.0,7.8383,1
9572,-122.5,37.6,35.0,2197.0,369.0,971.0,326.0,4.25,0
4653,-118.21,34.11,32.0,2759.0,499.0,1661.0,533.0,4.3812,1
14369,-118.33,33.91,39.0,1224.0,312.0,1106.0,333.0,3.3491,1


In [560]:
normalizer = Normalizer()
x_train_norm = normalizer.fit_transform(x_train)
x_test_norm = normalizer.transform(x_test)

In [561]:
from_numpy(y_train.values.astype(np.float32))

tensor([216300., 178600.,  74700.,  ..., 228200., 181800., 225900.])

In [562]:
x_train_tensor = from_numpy(x_train_norm).to(float32)
x_test_tensor = from_numpy(x_test_norm).to(float32)
y_train_tensor = from_numpy(y_train.values.astype(np.float32))
y_test_tensor = from_numpy(y_test.values.astype(np.float32))

In [563]:
class NeuralNetwork(nn.Module):
	def __init__(self):
		super(NeuralNetwork, self).__init__()
		self.flatten = nn.Flatten()
		self.layer_stack = nn.Sequential(
			nn.Linear(9, 1)
		)

	def forward(self, x):
		x = self.flatten(x)
		logits = self.layer_stack(x)
		return logits

model = NeuralNetwork().to('cpu')
print(model)
# list(model.named_parameters())
# model.layer_stack[0].weight.data

NeuralNetwork(
  (flatten): Flatten(start_dim=1, end_dim=-1)
  (layer_stack): Sequential(
    (0): Linear(in_features=9, out_features=1, bias=True)
  )
)


In [564]:
criterion = nn.MSELoss()
optimiser = optim.SGD(model.parameters(), lr = 0.001)

In [565]:
# def train_loop(features, labels, model, loss_fn, optimiser):
# 	dataset = TensorDataset(features, labels)
# 	dl = DataLoader(dataset)

# 	size = len(dataset)
# 	# for batch, (X, y) in enumerate(dl):
# 	# 	y_hat = model(X)
# 	# 	loss = loss_fn(y_hat, y)

# 	# 	optimiser.zero_grad()
# 	# 	loss.backward()
# 	# 	optimiser.step()

# 	# 	if batch % 100 == 0:
# 	# 		current = batch * len(X)
# 	# 		print(f"loss: {loss.item() ** 0.5} [{current}/{size}]")
# 	y_hat = model(features)
# 	loss = loss_fn(y_hat, labels)

# 	optimiser.zero_grad()
# 	loss.backward()
# 	optimiser.step()

# 	print(f"loss")

# train_loop(x_train_tensor, y_train_tensor, model, criterion, optimiser)


In [566]:
model(x_train_tensor)
# reshape(model(x_train_tensor),(len(x_train_tensor),))

tensor([[-0.3218],
        [-0.3792],
        [-0.3286],
        ...,
        [-0.3495],
        [-0.3719],
        [-0.3318]], grad_fn=<AddmmBackward0>)

In [567]:
reshape(model(x_train_tensor), (-1,))

tensor([-0.3218, -0.3792, -0.3286,  ..., -0.3495, -0.3719, -0.3318],
       grad_fn=<ReshapeAliasBackward0>)

In [568]:

n_epochs = 5000
for epoch in range(n_epochs):
	print(f"Epoch {epoch + 1}\n-------------------------------")
	y_hat = reshape(model(x_train_tensor), (-1,))
	loss = criterion(y_hat, y_train_tensor)

	optimiser.zero_grad()
	loss.backward()
	optimiser.step()

	print(f"loss: {loss}")

Epoch 1
-------------------------------
loss: 56214315008.0
Epoch 2
-------------------------------
loss: 55877419008.0
Epoch 3
-------------------------------
loss: 55543173120.0
Epoch 4
-------------------------------
loss: 55211556864.0
Epoch 5
-------------------------------
loss: 54882529280.0
Epoch 6
-------------------------------
loss: 54556086272.0
Epoch 7
-------------------------------
loss: 54232199168.0
Epoch 8
-------------------------------
loss: 53910867968.0
Epoch 9
-------------------------------
loss: 53592051712.0
Epoch 10
-------------------------------
loss: 53275734016.0
Epoch 11
-------------------------------
loss: 52961902592.0
Epoch 12
-------------------------------
loss: 52650524672.0
Epoch 13
-------------------------------
loss: 52341604352.0
Epoch 14
-------------------------------
loss: 52035092480.0
Epoch 15
-------------------------------
loss: 51730989056.0
Epoch 16
-------------------------------
loss: 51429273600.0
Epoch 17
------------------------

In [569]:
13353574400 > 12765857792

True

In [570]:
model(x_train_tensor)

tensor([[208924.3906],
        [182800.1250],
        [213955.0625],
        ...,
        [207353.6094],
        [194474.3125],
        [215722.5781]], grad_fn=<AddmmBackward0>)

In [571]:
y_train_tensor

tensor([216300., 178600.,  74700.,  ..., 228200., 181800., 225900.])

In [573]:
y

0        164600.0
1        125000.0
2        244500.0
3        106300.0
4        265800.0
           ...   
16507    111500.0
16508    238700.0
16509    275000.0
16510    150400.0
16511    133900.0
Name: median_house_value, Length: 16512, dtype: object

In [575]:
features = df.drop("median_house_value", axis=1)
labels = df["median_house_value"]

In [581]:
from part2_house_value_regression import Regressor

regressor = Regressor(features)

regressor.fit(features, labels)

TypeError: Regressor.__init__() missing 1 required positional argument: 'y'